# 02 — Benchmark 12 Unitree G1 tasks

Compare **your model** vs **Unitree VLA** (UnifoLM-VLA) on the 12 manipulation tasks.

This notebook imports existing S2R helpers and writes scores under `data/benchmark/`.

In [ ]:
from pathlib import Path
import sys
sys.path.insert(0, str(Path.cwd() / "_lib"))
from bootstrap import setup
ROOT = setup()

import pandas as pd
from s2r.experiments.paths import ensure_experiment_dirs, BENCHMARK_RESULTS
from s2r.experiments.benchmark import (
    load_tasks, score_episode, save_score, summarize_results, make_leaderboard
)

ensure_experiment_dirs()
tasks = load_tasks()
print(len(tasks), "tasks")
pd.DataFrame([{"id": t.id, "name": t.name, "hf": t.hf_dataset} for t in tasks])

## Record an evaluation episode

After you run a real/sim trial (pipeline deploy or Unitree VLA), fill the metrics below and save.

Tip: live pipeline metrics are also in the GUI and `data/raw/episode_*.jsonl`.

In [ ]:
MODEL_NAME = "my_r2s_agent"   # or "unitree_unifolm_vla"
TASK_ID = "05_pack_pencilbox"  # change per run
EPISODE_ID = "ep001"

score = score_episode(
    task_id=TASK_ID,
    model_name=MODEL_NAME,
    episode_id=EPISODE_ID,
    success=True,
    completion_time_s=42.0,
    interventions=0,
    e2e_latency_ms=85.0,
    vla_hz=2.0,
    esn_hz=100.0,
    decision_latency_ms=35.0,
    extras={"notes": "example score from notebook"},
)
path = save_score(score)
print("saved", path)

In [ ]:
# Demo: seed a few placeholder comparisons so the leaderboard renders
import itertools
demo = []
for model, base in [("my_r2s_agent", 0.55), ("unitree_unifolm_vla", 0.75)]:
    for i, t in enumerate(tasks):
        # deterministic pseudo scores for scaffolding plots only
        ok = ((i * 3 + len(model)) % 5) != 0 if model.startswith("my_") else ((i + 1) % 7) != 0
        s = score_episode(
            task_id=t.id,
            model_name=model,
            episode_id="demo",
            success=ok,
            completion_time_s=30 + i * 2,
            e2e_latency_ms=70 + (0 if "unitree" in model else 20),
            vla_hz=2.0,
            esn_hz=100.0 if model.startswith("my_") else 0.0,
        )
        save_score(s)
        demo.append(s)
print("demo scores", len(demo))

In [ ]:
summary = pd.DataFrame(summarize_results())
summary.sort_values(["task_id", "model"])

In [ ]:
import matplotlib.pyplot as plt

pivot = summary.pivot_table(index="task_id", columns="model", values="success_rate")
ax = pivot.plot(kind="bar", figsize=(12, 4))
ax.set_ylim(0, 1.05)
ax.set_ylabel("success rate")
ax.set_title("12-task benchmark: my model vs Unitree VLA")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

board = pd.DataFrame(make_leaderboard())
board

## Linking Hugging Face Unitree datasets

Each task maps to an official dataset (see `data/benchmark/tasks.yaml`), e.g.
`unitreerobotics/G1_Pack_PencilBox`.

```python
# optional
# from datasets import load_dataset
# ds = load_dataset("unitreerobotics/G1_Stack_Block")
```

Store converted episodes under `data/benchmark/tasks/<task_id>/episodes/`.